Step 3: GA implementation with DEAP

In [2]:
import random
import numpy as np
import pandas as pd
import joblib

from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty

from deap import base, creator, tools, algorithms


In [3]:
import joblib
import pandas as pd
import os

# Load pkl

rf_model_directory = "../ml"
rf_model_filename = "rf_bandgap_dp.pkl"
rf_model_path = os.path.join(rf_model_directory, rf_model_filename)

rf_model = joblib.load(rf_model_path)

# Load feature column 
ftr_col_directory = "../ml"
ftr_col_filename = "feature_columns.pkl"
ftr_col_path = os.path.join(ftr_col_directory, ftr_col_filename)

FEATURE_COLUMNS = joblib.load(ftr_col_path)
print(f"Loaded {len(FEATURE_COLUMNS)} feature columns for GA.")

ep = ElementProperty.from_preset("magpie")

Loaded 132 feature columns for GA.


/home/mostaq/anaconda3/lib/python3.11/site-packages/matminer/utils/data.py:326: UserWarning: MagpieData(impute_nan=False):
In a future release, impute_nan will be set to True by default.
                    This means that features that are missing or are NaNs for elements
                    from the data source will be replaced by the average of that value
                    over the available elements.
                    This avoids NaNs after featurization that are often replaced by
                    dataset-dependent averages.
  warnings.warn(f"{self.__class__.__name__}(impute_nan=False):\n" + IMPUTE_NAN_WARNING)


In [4]:
# A-site pool (alkaline-earth + rare-earth)
A_SITE = [
    "Ba", "Sr", "Ca", "Mg", "Li", "Na", "K", "Rb", "Cs",
    "La", "Ce", "Pr", "Nd", "Sm", "Gd", "Y", "Bi"
]

# B-site pool (ransition metals for perovskites)
B_SITE = [
    "Fe", "Mn", "Co", "Ni", "Ti", "Cr", "V", "Cu", "Zn", "Mo", "W"
]

# B' site pool (ifferent transition metals)
B_PRIME_SITE = [
    "Nb", "Ta", "Mo", "W", "V", "Ti", "Cr", "Re"
]


In [5]:
O_RADIUS = 1.40  # Å

def tolerance_factor(A, B, Bp):
    rA = Element(A).ionic_radii[2]
    rB = Element(B).ionic_radii[6]
    rBp = Element(Bp).ionic_radii[6]
    rB_avg = 0.5 * (rB + rBp)
    return (rA + O_RADIUS) / (np.sqrt(2) * (rB_avg + O_RADIUS))

def octahedral_factor(B, Bp):
    rB = Element(B).ionic_radii[6]
    rBp = Element(Bp).ionic_radii[6]
    return 0.5 * (rB + rBp) / O_RADIUS


In [6]:
def featurize_formula(formula):
    comp = Composition(formula)
    df = pd.DataFrame({"composition": [comp]})

    X = ep.featurize_dataframe(df, "composition", ignore_errors=True)
    X = X.select_dtypes(include="number")

    # align feature columns
    for col in FEATURE_COLUMNS:
        if col not in X:
            X[col] = 0.0

    X = X[FEATURE_COLUMNS]
    return X


In [7]:
def build_formula(ind):
    A = A_SITE[ind[0]]
    B = B_SITE[ind[1]]
    Bp = B_PRIME_SITE[ind[2]]
    return f"{A}2{B}{Bp}O6"


In [8]:
TARGET_EG = 1.5

def evaluate(ind):
    A = A_SITE[ind[0]]
    B = B_SITE[ind[1]]
    Bp = B_PRIME_SITE[ind[2]]

    formula = build_formula(ind)

    try:
        X = featurize_formula(formula)
        Eg = rf_model.predict(X)[0]
    except Exception:
        return 10.0, 10.0

    Eg_error = abs(Eg - TARGET_EG)

    try:
        t = tolerance_factor(A, B, Bp)
        mu = octahedral_factor(B, Bp)

        penalty = 0.0
        if not (0.8 <= t <= 1.05):
            penalty += abs(t - 0.95)
        if not (0.41 <= mu <= 0.9):
            penalty += abs(mu - 0.6)

    except Exception:
        penalty = 10.0

    return Eg_error, penalty


In [9]:
creator.create("FitnessMulti", base.Fitness, weights=(-1.0, -1.0))
creator.create("Individual", list, fitness=creator.FitnessMulti)

toolbox = base.Toolbox()

toolbox.register("A_idx", random.randrange, len(A_SITE))
toolbox.register("B_idx", random.randrange, len(B_SITE))
toolbox.register("Bp_idx", random.randrange, len(B_PRIME_SITE))

toolbox.register(
    "individual",
    tools.initCycle,
    creator.Individual,
    (toolbox.A_idx, toolbox.B_idx, toolbox.Bp_idx),
    n=1
)

toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate)
toolbox.register("mate", tools.cxUniform, indpb=0.5)
toolbox.register("mutate", tools.mutUniformInt,
                 low=[0,0,0],
                 up=[len(A_SITE)-1, len(B_SITE)-1, len(B_PRIME_SITE)-1],
                 indpb=0.2)
toolbox.register("select", tools.selNSGA2)


In [13]:
POP_SIZE = 120
NGEN = 20

pop = toolbox.population(n=POP_SIZE)

# ✅ Evaluate initial population
invalid_ind = [ind for ind in pop if not ind.fitness.valid]
fits = map(toolbox.evaluate, invalid_ind)
for ind, fit in zip(invalid_ind, fits):
    ind.fitness.values = fit

pop = toolbox.select(pop, len(pop))

for gen in range(NGEN):
    offspring = algorithms.varAnd(pop, toolbox, cxpb=0.7, mutpb=0.3)

    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fits = map(toolbox.evaluate, invalid_ind)
    for ind, fit in zip(invalid_ind, fits):
        ind.fitness.values = fit

    pop = toolbox.select(pop + offspring, POP_SIZE)

    front = tools.sortNondominated(pop, k=len(pop), first_front_only=True)[0]
    best = front[0]

    print(
        f"Gen {gen:02d} | "
        f"|Eg-1.5|={best.fitness.values[0]:.3f} | "
        f"penalty={best.fitness.values[1]:.3f} | "
        f"{build_formula(best)}"
    )


ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

...

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

Gen 19 | |Eg-1.5|=0.000 | penalty=10.000 | Y2FeTaO6


In [14]:
pareto_front = tools.sortNondominated(pop, k=len(pop), first_front_only=True)[0]

results = []
for ind in pareto_front:
    A, B, Bp = A_SITE[ind[0]], B_SITE[ind[1]], B_PRIME_SITE[ind[2]]
    results.append((f"{A}2{B}{Bp}O6", ind.fitness.values[0], ind.fitness.values[1]))

df_pareto = pd.DataFrame(
    results,
    columns=["formula", "predicted_bandgap", "stability_penalty"]
).drop_duplicates().sort_values(
    ["stability_penalty","predicted_bandgap"], ascending=[True, False]
)

df_pareto.head(10)


,formula,predicted_bandgap,stability_penalty
0,Y2FeTaO6,0.000101,10.0
